In [3]:
from IPython.display import HTML
with open("custom/style.css", "r") as f:
    custom_style = f.read()
hide_cell_style = "<style>.jp-Cell:has(.hide-me) { display: yes; }</style>"
HTML(f"<div class='hide-me'></div><style>{custom_style}</style>{hide_cell_style}")

<h1><center><font color='blue'>Introduction to Parallel Programming with MPI<br><br>Collective Operations</font></center></h1>

# <font color='blue'>Collective Operations</font>

## Motivation

<font color='green'>Dot product</font>: basic reduction using MPI collectives <br>
The inner product of two vectors: $\;\;\vec{a}.\vec{b}=\sum_{i=1}^N a_i b_i$ <br>
It can be calculated with the following loop in C programming language:
```cpp
dot=0.0;
for(int i=0;i<N;i++) dot+=a[i]*b[i];
```
dividing work across $N_{\text{proc}}$ prcesses:
$N_l=N/N_{\text{proc}}$
```cpp
dot=0.0;
for(int i=0;i<Nl;i++) dot+=a[i]*b[i];
```
<!-- <img src="figs/dot-product.svg" alt="MPI-standard" align="right" width="35%"/><p> </p> -->
```{image} figs/dot-product.svg
:width: 1000px
```
- Each rank has indeed a variable `dot` carrying a portion of the dot product
- Gathering `dot`s onto a rank (say rank `0`) in array `dot_arr`, then `dot_arr[0]+dot_arr[1]+dot_arr[2]` <br>
<font size="6" color='green'>Is there a better solution?</font>

## Global operations: reduction

- Compute results over distributed data
```cpp
int MPI_Reduce(const void *sendbuf, void *recvbuf, int count,
               MPI_Datatype datatype, MPI_Op op, int root, MPI_Comm comm)
```
- Result in `recvbuf` only available on `root` process
- Perform operation on all `count` elements of an array
- If all ranks need the result, then use `MPI_Allreduce()`
- If the <font size='5' color='green'>12</font> predefined operations are not enough use `MPI_Op_create`/`MPI_Op_free` to create own ones
<!-- <img src="figs/global-operations.svg" alt="MPI-standard" align="right" width="35%"/><p> </p> -->
```{image} figs/global-operations.svg
:width: 400px
```

## Global operations - predefined operators

<table style="border: 1px solid black; text-align: center">
  <tr>
    <th style="text-align: center; background-color: powderblue">Name</th><th style="text-align: center; background-color: powderblue">Operation</th><th style="text-align: center; background-color: powderblue">Name</th><th style="text-align: center; background-color: powderblue">Operation</th>
  </tr>
    <tr><td style="text-align: left">MPI_SUM   </td><td style="text-align: left">Sum             </td><td style="text-align: left">MPI_PROD  </td><td style="text-align: left">Product         </td></tr>
    <tr><td style="text-align: left">MPI_MAX   </td><td style="text-align: left">Maximum         </td><td style="text-align: left">MPI_MIN   </td><td style="text-align: left">Minimum         </td></tr>
    <tr><td style="text-align: left">MPI_LAND  </td><td style="text-align: left">Logical AND     </td><td style="text-align: left">MPI_BAND  </td><td style="text-align: left">Bit-AND         </td></tr>
    <tr><td style="text-align: left">MPI_LOR   </td><td style="text-align: left">Logical OR      </td><td style="text-align: left">MPI_BOR   </td><td style="text-align: left">Bit-OR          </td></tr>
    <tr><td style="text-align: left">MPI_LXOR  </td><td style="text-align: left">Logical XOR     </td><td style="text-align: left">MPI_BXOR  </td><td style="text-align: left">Bit-XOR         </td></tr>
    <tr><td style="text-align: left">MPI_MAXLOC</td><td style="text-align: left">Maximum+Position</td><td style="text-align: left">MPI_MINLOC</td><td style="text-align: left">Minimum+Position</td></tr>
</table>

- Define own operations with `MPI_Op_create`/`MPI_Op_free`
- MPI assumes that the operations are associative
  - be careful with floating-point operations

## In-place buffer specification

Override local input buffer with a result

- MPI_Reduce
```cpp
int partial_sum = ..., total_sum;
MPI_Reduce(&partial_sum, &total_sum, 1, MPI_INT, MPI_SUM, root, comm);

int partial_sum = ..., total_sum;
if (rank == root) {
    total_sum = partial_sum;
    MPI_Reduce(MPI_IN_PLACE, &total_sum, 1, MPI_INT, MPI_SUM, root, comm);
}
else {
    MPI_Reduce(&partial_sum, &total_sum, 1, MPI_INT, MPI_SUM, root, comm);
}
```
- MPI_Allreduce
```cpp
int partial_sum = ..., total_sum;
MPI_AllReduce(&partial_sum, &total_sum, 1, MPI_INT, MPI_SUM, comm);

int partial_sum = ..., total_sum;

total_sum = partial_sum;
MPI_AllReduce(MPI_IN_PLACE, &total_sum, 1, MPI_INT, MPI_SUM, comm);
```

## MPI_IN_PLACE cheat sheet

<div style="text-align: center; font-size: 24px; max-width: 800px;">
<span">
<table style="border: 1px solid black; text-align: center">
  <tr>
    <th style="text-align: center; background-color: powderblue">Function</th><th style="text-align: center; background-color: powderblue">MPI_IN_PLACE argument</th><th style="text-align: center; background-color: powderblue">@ rank(s)</th><th style="text-align: center; background-color: powderblue">Comment [MPI 3.0]</th>
  </tr>
    <tr><td style="text-align: left">MPI_GATHER    </td><td style="text-align: left">send buffer   </td><td style="text-align: left">root</td><td style="text-align: left">Recv value at root already in the correct place in receive buffer.                                                                                                                                                                             </td></tr>
    <tr><td style="text-align: left">MPI_GATHERV   </td><td style="text-align: left">send buffer   </td><td style="text-align: left">root</td><td style="text-align: left">Recv value at root already in the correct place in receive buffer.                                                                                                                                                                             </td></tr>
    <tr><td style="text-align: left">MPI_SCATTER   </td><td style="text-align: left">receive buffer</td><td style="text-align: left">root</td><td style="text-align: left">Root-thsegment of send buffer is not moved.                                                                                                                                                                                                   </td></tr>
    <tr><td style="text-align: left">MPI_SCATTERV  </td><td style="text-align: left">receive buffer</td><td style="text-align: left">root</td><td style="text-align: left">Root-thsegment of send buffer is not moved.                                                                                                                                                                                                   </td></tr>
    <tr><td style="text-align: left">MPI_ALLGATHER </td><td style="text-align: left">send buffer   </td><td style="text-align: left">all </td><td style="text-align: left">Input data at the correct place were process would receive its own contribution.                                                                                                                                                              </td></tr>
    <tr><td style="text-align: left">MPI_ALLGATHERV</td><td style="text-align: left">send buffer   </td><td style="text-align: left">all </td><td style="text-align: left">Input data at the correct place were process would receive its own contribution.                                                                                                                                                              </td></tr>
    <tr><td style="text-align: left">MPI_ALLTOALL  </td><td style="text-align: left">send buffer   </td><td style="text-align: left">all </td><td style="text-align: left">Data to be sent is taken from receive buffer and replaced by received data, data sent/received must be of the same type map specified in receive count/receive type.                                                                          </td></tr>
    <tr><td style="text-align: left">MPI_ALLTOALLV </td><td style="text-align: left">send buffer   </td><td style="text-align: left">all </td><td style="text-align: left">Data to be sent is taken from receive buffer and replaced by received data. Data sent/received must be of the same type map specified in receive count/receive type. The same amount of data and data type is exchanged between two processes.</td></tr>
    <tr><td style="text-align: left">MPI_REDUCE    </td><td style="text-align: left">send buffer   </td><td style="text-align: left">root</td><td style="text-align: left">Data taken from receive buffer, replaced with output data.                                                                                                                                                                                    </td></tr>
    <tr><td style="text-align: left">MPI_ALLREDUCE </td><td style="text-align: left">send buffer   </td><td style="text-align: left">all </td><td style="text-align: left">Data taken from receive buffer, replaced with output data.                                                                                                                                                                                    </td></tr>
</table>
</span>
</div>

## Summary of MPI collective communication

- MPI (blocking) <font color='green'>collectives</font>
  - All ranks in communicator <font color='green'>must call</font> the function
- Communication and synchronization
  - Barrier, broadcast, scatter, gather, and combinations thereof
- Global <font color='green'>operations</font>
  - <font color='green'>Reduce</font>, <font color='green'>allreduce</font>, some more ...
- <font color='green'>In-place buffer</font> specification  `MPI_IN_PLACE`
  - Save some space if needed

## Quiz:

1. Do you recommend an application developer to use `MPI_Gather` followed by operations on the <font color='green'>root</font> process rather than using `MPI_Reduce`? Why?
   <ol style="list-style-type: lower-alpha;">
   <li>Yes</li>
   <li>No</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> b., because it is very likely that it is slower than <code>MPI_Reduce</code>.
   </details> <p></p>
  
2. Is the argument `recvbuf` in `MPI_Reduce` <font color='green'>significant on every process</font> calling it?
   <ol style="list-style-type: lower-alpha;">
   <li>Yes</li>
   <li>No</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> b., it is significant only at root.
   </details> <p></p>